<a href="https://www.kaggle.com/code/lakshmibhavaniyepuri/depth-estimation?scriptVersionId=311608688" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports
<div style="width:100%;height:0;border-bottom: 3px solid #F59A31;margin-bottom: 1rem;"></div>


In [ ]:
!pip install -q "huggingface-hub<1.0.0" segmentation-models-pytorch albumentations torchmetrics timm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import cv2 as cv
from PIL import Image
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
from torchmetrics.regression import MeanSquaredError as MSE
from torchmetrics.collections import MetricCollection
import gc
from torchvision.transforms import Normalize
import math


# Data & Visualization
### Note: brighter the area, farther it is in the image, brighter => more depth
<div style="width:100%;height:0;border-bottom: 3px solid #F7D735;margin-bottom: 1rem;"></div>


In [ ]:
train_csv = Path('/kaggle/input/nyu-depth-v2/nyu_data/data/nyu2_train.csv')
base_path = Path('/kaggle/input/nyu-depth-v2/nyu_data')

df = pd.read_csv(train_csv, header=None)
df[0] = df[0].map(lambda x: base_path / x)
df[1] = df[1].map(lambda x: base_path / x)

train_df, val_df = train_test_split(df, test_size=0.1, shuffle=True, random_state=42)
val_df,  test_df = train_test_split(val_df, test_size=0.1, shuffle=True, random_state=42)
for d in [train_df, val_df, test_df]:
    d.reset_index(drop=True, inplace=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
def colored_depthmap(depth, d_min=None, d_max=None, cmap=plt.cm.inferno):
    d_min = d_min if d_min is not None else depth.min()
    d_max = d_max if d_max is not None else depth.max()
    rel   = (depth - d_min) / (d_max - d_min + 1e-8)
    return (255 * cmap(rel)[:, :, :3]).astype(np.uint8)

def merge_into_row(rgb, depth):
    rgb   = np.array(rgb)
    depth = np.squeeze(np.array(depth))
    return np.hstack([rgb, colored_depthmap(depth)])

plt.figure(figsize=(15, 6))
for i, idx in enumerate(np.random.randint(0, len(df), (16,))):
    ax = plt.subplot(4, 4, i + 1)
    image = Image.open(df.iloc[idx, 0]).convert('RGB')
    mask  = Image.open(df.iloc[idx, 1]).convert('L')
    plt.imshow(merge_into_row(image, mask))
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AUGMENTATIONS
#   • NYU Depth v2 depth maps are uint16 PNGs  → we keep float32 in [0,1]
#   • Image size 480×640 → crop/resize to 384 for training, 480 for val/test
# ─────────────────────────────────────────────────────────────────────────────
IMG_SIZE = 384
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

sample_tfms = [
    A.HorizontalFlip(p=0.5),
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.6, 1.0)),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
    A.OneOf([
        A.MotionBlur(p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.5),
    ], p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.HueSaturationValue(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
]

train_tfms = A.Compose([
    *sample_tfms,
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])
valid_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])

# Dataset
<div style="width:100%;height:0;border-bottom: 3px solid #F7F437;margin-bottom: 1rem;"></div>


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATASET
#   NYU Depth v2 depth PNGs are 16-bit; cv2 reads them as uint16.
#   Max valid depth = 10 m.  We normalise to [0, 1] by dividing by 10000
#   (values are in millimetres when read as uint16).
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE = 384
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

sample_tfms = [
    A.HorizontalFlip(p=0.5),
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.6, 1.0)),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
    A.OneOf([
        A.MotionBlur(p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.5),
    ], p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.HueSaturationValue(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
]

train_tfms = A.Compose([
    *sample_tfms,
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])
valid_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADERS
# ─────────────────────────────────────────────────────────────────────────────
BATCH = 16
NUM_W = 4

train_dl = torch.utils.data.DataLoader(
    train_df, shuffle=True,  batch_size=BATCH, num_workers=NUM_W,
    pin_memory=True, drop_last=True)
val_dl   = torch.utils.data.DataLoader(
    val_df,   shuffle=False, batch_size=BATCH, num_workers=NUM_W,
    pin_memory=True)
test_dl  = torch.utils.data.DataLoader(
    test_df,  shuffle=True,  batch_size=4, num_workers=NUM_W,
    pin_memory=True)

print(f"Train batches: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADERS
# ─────────────────────────────────────────────────────────────────────────────
BATCH = 16
NUM_W = 4

train_ds = DepthDataset(train_df, transforms=train_tfms)
val_ds   = DepthDataset(val_df,   transforms=valid_tfms)
test_ds  = DepthDataset(test_df,  transforms=valid_tfms)

train_dl = torch.utils.data.DataLoader(
    train_ds, shuffle=True,  batch_size=BATCH, num_workers=NUM_W,
    pin_memory=True, drop_last=True)
val_dl   = torch.utils.data.DataLoader(
    val_ds,   shuffle=False, batch_size=BATCH, num_workers=NUM_W,
    pin_memory=True)
test_dl  = torch.utils.data.DataLoader(
    test_ds,  shuffle=False, batch_size=4, num_workers=NUM_W,  # ← also fixed shuffle=True→False for test
    pin_memory=True)

print(f"Train batches: {len(train_dl)} | Val: {len(val_dl)} | Test: {len(test_dl)}")

# Model
<div style="width:100%;height:0;border-bottom: 3px solid #C3F93B;margin-bottom: 1rem;"></div>

In [ ]:
ENCODER     = 'efficientnet-b5'
ENCODER_WTS = 'imagenet'

class DepthModel(nn.Module):
    def __init__(self, encoder=ENCODER, weights=ENCODER_WTS):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type='scse',
        )

    def set_encoder_trainable(self, trainable: bool):
        for p in self.model.encoder.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        return torch.sigmoid(self.model(x))

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
class ScaleInvariantLogLoss(nn.Module):
    def __init__(self, lam=0.85, eps=1e-6):
        super().__init__()
        self.lam = lam
        self.eps = eps

    def forward(self, pred, target):
        mask = (target > self.eps).float()
        n    = mask.sum().clamp(min=1)
        d    = (torch.log(pred.clamp(min=self.eps)) -
                torch.log(target.clamp(min=self.eps))) * mask
        loss = (d**2).sum() / n - self.lam * (d.sum() / n) ** 2
        return torch.sqrt(loss.clamp(min=0))


class SSIMLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self._ssim = SSIM(data_range=1.0).to('cuda')

    def forward(self, pred, target):
        return 1.0 - self._ssim(pred, target)


class CombinedDepthLoss(nn.Module):
    def __init__(self, alpha=0.8, beta=0.1, gamma=0.1):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.silog  = ScaleInvariantLogLoss()
        self.ssim_l = SSIMLoss()

    @staticmethod
    def _gradient_loss(pred, target):
        def grad(t):
            dx = t[:, :, :, 1:] - t[:, :, :, :-1]
            dy = t[:, :, 1:, :] - t[:, :, :-1, :]
            return dx, dy
        px, py = grad(pred)
        tx, ty = grad(target)
        return F.l1_loss(px, tx) + F.l1_loss(py, ty)

    def forward(self, pred, target):
        return (self.alpha * self.silog(pred, target)
              + self.beta  * self.ssim_l(pred, target)
              + self.gamma * self._gradient_loss(pred, target))

# Training
<div style="width:100%;height:0;border-bottom: 3px solid #F59A31;margin-bottom: 1rem;"></div>

### Metrics:

####  Structural Similarity Index (SSIM)

The structural similarity index (SSIM) is a perceptual metric that takes into account the structural similarity of the two images. SSIM is calculated by comparing the local patterns of the two images, taking into account the luminance, contrast, and structure of the images.

The formula for SSIM is:

$$ SSIM = \frac{(2 \mu_x \mu_y + c_1) (2 \sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1) (\sigma_x^2 + \sigma_y^2 + c_2)} $$

where μx​ and μy​ are the means of the two images, σx2​ and σy2​ are the variances of the two images, and σxy​ is the covariance of the two images. c1​ and c2​ are constants that are used to stabilize the calculation of SSIM.

SSIM is more robust to noise and small changes than MSE, but it is also more computationally expensive to calculate.

In general, SSIM is preferred over MSE for image quality assessment because it provides a more accurate measure of how humans perceive the similarity of two images. However, MSE is still a useful metric, especially when speed is important.

In [ ]:
class UnNormalize(Normalize):
    def __init__(self):
        new_mean = [-m / s for m, s in zip(MEAN, STD)]
        new_std  = [1.0 / s for s in STD]
        super().__init__(new_mean, new_std)

@torch.no_grad()
def plot_predictions(imgs, preds, targets, n=4, figsize=(8, 4), title=''):
    unnorm = UnNormalize()
    n      = min(n, imgs.size(0))
    plt.figure(figsize=figsize, dpi=150)
    for i in range(n):
        img = unnorm(imgs[i]).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        pre = preds[i].squeeze().cpu().numpy()
        gt  = targets[i].squeeze().cpu().numpy()
        vis = np.hstack([
            (img * 255).astype(np.uint8),
            colored_depthmap(pre),
            colored_depthmap(gt),
        ])
        ax = plt.subplot(1, n, i + 1)
        plt.imshow(vis)
        plt.axis("off")
    t = f"{title}\n[RGB | Pred | GT]" if title else "[RGB | Pred | GT]"
    plt.suptitle(t, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALISATION
# ─────────────────────────────────────────────────────────────────────────────
 
@torch.no_grad()
def plot_predictions(imgs, preds, targets, n=4, figsize=(8, 2), title=''):
    unnorm = UnNormalize()
    n      = min(n, imgs.size(0))
    r      = (n + 1) // 2
    plt.figure(figsize=figsize, dpi=150)
    for i, idx in enumerate(np.random.randint(0, imgs.size(0), (n,))):
        ax  = plt.subplot(r, (n + r - 1) // r * 2, i + 1)
        img = unnorm(imgs[idx]).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        pre = preds[idx].squeeze().cpu().numpy()
        gt  = targets[idx].squeeze().cpu().numpy()
        vis = np.hstack([
            (img * 255).astype(np.uint8),
            colored_depthmap(pre),
            colored_depthmap(gt),
        ])
        plt.imshow(vis)
        plt.axis("off")
    t = f"{title}\n[RGB | Prediction | Ground Truth]" if title else "[RGB | Prediction | Ground Truth]"
    plt.suptitle(t, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
EPOCHS        = 15
FREEZE_EPOCHS = 3
LR            = 3e-4
WEIGHT_DECAY  = 1e-2
DEVICE        = 'cuda'
SAVE_PATH     = 'depth_efficientnet_b5_unetplusplus.pt'

# Reinitialise metrics
metrics = MetricCollection([
    SSIM(data_range=(0, 1)),
    MSE(),
]).to(DEVICE)
train_metrics = metrics.clone(prefix='train_')
val_metrics   = metrics.clone(prefix='val_')

# Reinitialise model fresh
model = DepthModel().to(DEVICE)
model.set_encoder_trainable(False)
print(f"Trainable parameters: {model.n_params:,}")

loss_fn = CombinedDepthLoss().to(DEVICE)

optim = torch.optim.AdamW([
    {'params': model.model.decoder.parameters(),           'lr': LR},
    {'params': model.model.segmentation_head.parameters(), 'lr': LR},
    {'params': model.model.encoder.parameters(),           'lr': LR / 10,
     'weight_decay': WEIGHT_DECAY},
], weight_decay=WEIGHT_DECAY)

def lr_lambda(step):
    warmup_steps = FREEZE_EPOCHS * len(train_dl)
    total_steps  = EPOCHS * len(train_dl)
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

sched  = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)
scaler = GradScaler()   # fresh scaler

logs = pd.DataFrame(columns=['loss_train','loss_val','ssim_train','ssim_val','mse_train','mse_val'])
best_ssim  = -1e9
best_epoch = -1

gc.collect()
torch.cuda.empty_cache()
print("All reinitialised OK")

In [ ]:
print("Phase 1: training decoder only ...")

for epoch in tqdm(range(EPOCHS), ncols=80, ascii=True):

    if epoch == FREEZE_EPOCHS:
        model.set_encoder_trainable(True)
        print(f"Epoch {epoch}: unfreezing encoder ...")

    # ── TRAIN ──
    model.train()
    running_loss = 0.0

    for batch_idx, (img, mask) in enumerate(train_dl):
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        optim.zero_grad(set_to_none=True)

        with autocast():
            preds = model(img)
            loss  = loss_fn(preds, mask)

        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
        sched.step()

        running_loss += loss.item()
        train_metrics(preds.detach(), mask)
        del img, mask, preds, loss

        if batch_idx % 100 == 0:
            print(f"  Epoch {epoch} | step {batch_idx}/{len(train_dl)} | avg_loss {running_loss/(batch_idx+1):.4f}", flush=True)

    m = train_metrics.compute()
    logs.loc[epoch, ['loss_train','ssim_train','mse_train']] = (
        running_loss / len(train_dl),
        m['train_StructuralSimilarityIndexMeasure'].item(),
        m['train_MeanSquaredError'].item(),
    )
    train_metrics.reset()

    # ── VALIDATE ──
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for img, mask in val_dl:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            with autocast():
                preds = model(img)
                loss  = loss_fn(preds, mask)
            running_loss += loss.item()
            val_metrics(preds, mask)
            del img, mask, preds, loss

    m = val_metrics.compute()
    _ssim = m['val_StructuralSimilarityIndexMeasure'].item()
    _mse  = m['val_MeanSquaredError'].item()
    logs.loc[epoch, ['loss_val','ssim_val','mse_val']] = (
        running_loss / len(val_dl), _ssim, _mse)
    val_metrics.reset()

    # ── CHECKPOINT ──
    if _ssim > best_ssim:
        best_ssim, best_epoch = _ssim, epoch
        torch.save({
            'epoch':      epoch,
            'state_dict': model.state_dict(),
            'ssim':       _ssim,
            'mse':        _mse,
            'encoder':    ENCODER,
        }, SAVE_PATH)
        print(f"  Saved checkpoint (SSIM={_ssim:.4f})")

    print(f"\n{logs.tail(1).to_string()}\n", flush=True)

    # ── PREVIEW ──
    model.eval()
    with torch.no_grad():
        imgs_b, masks_b = next(iter(test_dl))
        with autocast():
            preds_b = model(imgs_b.to(DEVICE))
        plot_predictions(
            imgs_b.float().cpu(),
            preds_b.float().cpu(),
            masks_b.float().cpu(),
            n=4
        )
        del imgs_b, masks_b, preds_b

    gc.collect()
    torch.cuda.empty_cache()

print(f"\nBest epoch: {best_epoch}  |  Best val SSIM: {best_ssim:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION ON TEST SET (best checkpoint)
# ─────────────────────────────────────────────────────────────────────────────
SAVE_PATH="/kaggle/working/depth_efficientnet_b5_unetplusplus.pt"
ckpt = torch.load(SAVE_PATH)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (SSIM={ckpt['ssim']:.4f})")
 
all_imgs, all_preds, all_targets = [], [], []
model.eval()
with torch.no_grad(), autocast():
    for img, mask in tqdm(test_dl, total=len(test_dl)):
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        preds = model(img)
        all_imgs.append(img.cpu())
        all_preds.append(preds.cpu())
        all_targets.append(mask.cpu())
 
all_preds_t   = torch.vstack(all_preds)
all_targets_t = torch.vstack(all_targets)
 
test_m = metrics.clone().to('cpu')
test_m(all_preds_t, all_targets_t)
m      = test_m.compute()
title  = (f"Test  SSIM: {m['StructuralSimilarityIndexMeasure']:.4f} | "
          f"MSE: {m['MeanSquaredError']:.5f}")
print(title)
 
plot_predictions(
    torch.vstack(all_imgs),
    all_preds_t,
    all_targets_t,
    n=16, figsize=(12, 18), title=title,
)

In [ ]:
print(df.columns.tolist())
print(df.head(2))

In [ ]:
# ── wherever you load df, change this ────────────────────────────────────────
df = pd.read_csv('/kaggle/input/nyu-depth-v2/nyu_data/data/nyu2_train.csv',
                 header=None)          # ← header=None because file has no header row
df.columns = ['image_path', 'depth_path']  # ← add this line right after
print(df.shape)
print(df.head(2))

In [ ]:
BASE = '/kaggle/input/nyu-depth-v2/nyu_data/'

df.columns = ['image_path', 'depth_path']
df['image_path'] = BASE + df['image_path']
df['depth_path'] = BASE + df['depth_path']

print(df.head(2))  # verify full paths

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.cuda.amp import autocast
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# MODEL CLASS (SAME AS TRAINING)
# ─────────────────────────────────────────────────────────────────────────────
ENCODER     = 'efficientnet-b5'
ENCODER_WTS = 'imagenet'

class DepthModel(nn.Module):
    def __init__(self, encoder=ENCODER, weights=ENCODER_WTS):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type='scse',
        )

    def forward(self, x):
        return torch.sigmoid(self.model(x))

# ─────────────────────────────────────────────────────────────────────────────
# CREATE MODEL (IMPORTANT FIX)
# ─────────────────────────────────────────────────────────────────────────────
model = DepthModel().to(DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# LOAD CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
SAVE_PATH = "/kaggle/working/depth_efficientnet_b5_unetplusplus.pt"

ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])

print(f"Loaded checkpoint from epoch {ckpt['epoch']} (SSIM={ckpt['ssim']:.4f})")

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
all_imgs, all_preds, all_targets = [], [], []

model.eval()
with torch.no_grad():
    for img, mask in tqdm(test_dl, total=len(test_dl)):
        img = img.to(DEVICE)
        mask = mask.to(DEVICE)

        with autocast(enabled=(DEVICE.type == "cuda")):
            preds = model(img)

        all_imgs.append(img.cpu())
        all_preds.append(preds.cpu())
        all_targets.append(mask.cpu())

# STACK
all_preds_t   = torch.vstack(all_preds)
all_targets_t = torch.vstack(all_targets)

# METRICS
test_m = metrics.clone().to('cpu')
test_m(all_preds_t, all_targets_t)
m = test_m.compute()

title = (f"Test SSIM: {m['StructuralSimilarityIndexMeasure']:.4f} | "
         f"MSE: {m['MeanSquaredError']:.5f}")

print(title)

# VISUALIZATION
plot_predictions(
    torch.vstack(all_imgs),
    all_preds_t,
    all_targets_t,
    n=16,
    figsize=(12, 18),
    title=title,
)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast
from tqdm import tqdm
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import torchmetrics
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# LOAD & FIX DATAFRAME
# ─────────────────────────────────────────────────────────────────────────────
BASE = '/kaggle/input/nyu-depth-v2/nyu_data/'

df = pd.read_csv('/kaggle/input/nyu-depth-v2/nyu_data/data/nyu2_train.csv', header=None)
df.columns = ['image_path', 'depth_path']
df['image_path'] = BASE + df['image_path']
df['depth_path']  = BASE + df['depth_path']
print(f"Total samples: {len(df)}")

# ─────────────────────────────────────────────────────────────────────────────
# SPLIT
# ─────────────────────────────────────────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

# ─────────────────────────────────────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────
IMG_SIZE = 384
MEAN     = (0.485, 0.456, 0.406)
STD      = (0.229, 0.224, 0.225)

train_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.6, 1.0)),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
    A.OneOf([
        A.MotionBlur(p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.5),
    ], p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.HueSaturationValue(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])

valid_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])

# ─────────────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────────────
class DepthDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # RGB image
        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Depth mask — 16-bit PNG, normalise mm → [0, 1] at 10 m max
        mask = cv2.imread(row['depth_path'], cv2.IMREAD_UNCHANGED)
        mask = mask.astype(np.float32) / 10000.0
        mask = np.clip(mask, 0.0, 1.0)

        if self.transforms:
            augmented = self.transforms(image=img, mask=mask)
            img  = augmented['image']
            mask = augmented['mask']

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return img, mask

# ─────────────────────────────────────────────────────────────────────────────
# DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
train_ds = DepthDataset(train_df, transforms=train_tfms)
valid_ds = DepthDataset(valid_df, transforms=valid_tfms)
test_ds  = DepthDataset(test_df,  transforms=valid_tfms)

train_dl = DataLoader(train_ds, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
valid_dl = DataLoader(valid_ds, batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=4,  shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_dl)} | Valid: {len(valid_dl)} | Test: {len(test_dl)}")

# ─────────────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────────────
ENCODER     = 'efficientnet-b5'
ENCODER_WTS = 'imagenet'

class DepthModel(nn.Module):
    def __init__(self, encoder=ENCODER, weights=ENCODER_WTS):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type='scse',
        )

    def forward(self, x):
        return torch.sigmoid(self.model(x))

model = DepthModel().to(DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# LOAD CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
SAVE_PATH = "/kaggle/working/depth_efficientnet_b5_unetplusplus.pt"

ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (SSIM={ckpt['ssim']:.4f})")

# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
test_m = torchmetrics.MetricCollection({
    'StructuralSimilarityIndexMeasure': torchmetrics.StructuralSimilarityIndexMeasure(),
    'MeanSquaredError':                 torchmetrics.MeanSquaredError(),
}).to(DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION  (memory-safe)
# ─────────────────────────────────────────────────────────────────────────────
all_imgs, all_preds, all_targets = [], [], []
MAX_VIZ_BATCHES = 4   # only keep 16 images for visualization

model.eval()
with torch.no_grad():
    for i, (img, mask) in enumerate(tqdm(test_dl, total=len(test_dl))):
        img  = img.to(DEVICE)
        mask = mask.to(DEVICE)

        with autocast(enabled=(DEVICE.type == "cuda")):
            preds = model(img)

        # update metrics in-place on GPU
        test_m(preds, mask)

        # only save first MAX_VIZ_BATCHES for plotting
        if i < MAX_VIZ_BATCHES:
            all_imgs.append(img.cpu())
            all_preds.append(preds.cpu())
            all_targets.append(mask.cpu())

        del img, mask, preds
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# COMPUTE & PRINT METRICS
# ─────────────────────────────────────────────────────────────────────────────
m = test_m.compute()
title = (f"Test SSIM: {m['StructuralSimilarityIndexMeasure']:.4f} | "
         f"MSE: {m['MeanSquaredError']:.5f}")
print(title)

# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────
def plot_predictions(images, preds, targets, n=5, figsize=(18, 25), title=''):
    """Plot n samples: RGB image | predicted depth | ground truth depth"""
    images  = images[:n]
    preds   = preds[:n]
    targets = targets[:n]

    # denormalize RGB
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std  = torch.tensor(STD).view(3, 1, 1)
    images = images * std + mean
    images = images.permute(0, 2, 3, 1).numpy().clip(0, 1)

    preds   = preds.squeeze(1).numpy()
    targets = targets.squeeze(1).numpy()

    fig, axes = plt.subplots(n, 3, figsize=figsize)
    fig.suptitle(title, fontsize=14, y=1.01)

    for i in range(n):
        axes[i, 0].imshow(images[i])
        axes[i, 0].set_title("RGB", fontsize=13)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(preds[i], cmap='plasma')
        axes[i, 1].set_title("Predicted Depth", fontsize=13)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(targets[i], cmap='plasma')
        axes[i, 2].set_title("Ground Truth", fontsize=13)
        axes[i, 2].axis('off')

    plt.subplots_adjust(hspace=0.05, wspace=0.05)
    plt.savefig('/kaggle/working/predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → /kaggle/working/predictions.png")

plot_predictions(
    torch.vstack(all_imgs),
    torch.vstack(all_preds),
    torch.vstack(all_targets),
    n=5,
    figsize=(18, 25),   # wide + tall = big images
    title=title,
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
# ─────────────────────────────────────────────────────────────────────────────
# SPLIT DATAFRAME  (70% train | 15% valid | 15% test)
# ─────────────────────────────────────────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

# ─────────────────────────────────────────────────────────────────────────────
# INSTANTIATE DATASETS & DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
train_ds = DepthDataset(train_df, transforms=train_tfms)
valid_ds = DepthDataset(valid_df, transforms=valid_tfms)
test_ds  = DepthDataset(test_df,  transforms=valid_tfms)

train_dl = DataLoader(train_ds, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
class DepthDataset(torch.utils.data.Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)  # ← ensures clean 0-based index
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]  # ← .iloc fixes the KeyError: 215

        # ── RGB image ──────────────────────────────────────────────────────
        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ── Depth mask (16-bit PNG → float32 normalised to [0, 1]) ─────────
        mask = cv2.imread(row['depth_path'], cv2.IMREAD_UNCHANGED)  # uint16
        mask = mask.astype(np.float32) / 10000.0                    # mm → [0,1] @ 10 m max
        mask = np.clip(mask, 0.0, 1.0)

        # ── Albumentations expects HWC image and HW mask ───────────────────
        if self.transforms:
            augmented = self.transforms(image=img, mask=mask)
            img  = augmented['image']   # → CxHxW tensor (ToTensorV2)
            mask = augmented['mask']    # → HxW tensor

        # ── Model expects mask as (1, H, W) ───────────────────────────────
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return img, mask


# ─────────────────────────────────────────────────────────────────────────────
# INSTANTIATE DATASETS & DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
train_ds = DepthDataset(train_df, transforms=train_tfms)
valid_ds = DepthDataset(valid_df, transforms=valid_tfms)
test_ds  = DepthDataset(test_df,  transforms=valid_tfms)

train_dl = DataLoader(train_ds, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

# Results
<div style="width:100%;height:0;border-bottom: 3px solid #8DFA3F;margin-bottom: 1rem;"></div>

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
 
logs['loss_train'].plot(ax=axes[0], label='train')
logs['loss_val'].plot(ax=axes[0], label='val')
axes[0].set_title('Combined depth loss (↓)')
axes[0].legend()
 
logs['mse_train'].plot(ax=axes[1], label='train')
logs['mse_val'].plot(ax=axes[1], label='val')
axes[1].set_title('MSE (↓)')
axes[1].legend()
 
logs['ssim_train'].plot(ax=axes[2], label='train')
logs['ssim_val'].plot(ax=axes[2], label='val')
axes[2].set_title('SSIM (↑)')
axes[2].legend()
 
plt.tight_layout()
plt.show()

> visulisations

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import glob
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────────────
ENCODER     = 'efficientnet-b5'
ENCODER_WTS = 'imagenet'

class DepthModel(nn.Module):
    def __init__(self, encoder=ENCODER, weights=ENCODER_WTS):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type='scse',
        )

    def forward(self, x):
        return torch.sigmoid(self.model(x))

# ─────────────────────────────────────────────────────────────────────────────
# LOAD CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
PTH_PATH = "/kaggle/working/depth_efficientnet_b5_unetplusplus.pt"

model = DepthModel().to(DEVICE)
ckpt  = torch.load(PTH_PATH, map_location=DEVICE)

if 'state_dict' in ckpt:
    model.load_state_dict(ckpt['state_dict'])
    print(f"Loaded checkpoint — epoch {ckpt.get('epoch','?')} | "
          f"SSIM {ckpt.get('ssim', 0):.4f}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state_dict")

model.eval()

# ─────────────────────────────────────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────
IMG_SIZE = 384
MEAN     = (0.485, 0.456, 0.406)
STD      = (0.229, 0.224, 0.225)

infer_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD, always_apply=True),
    ToTensorV2(),
])

# ─────────────────────────────────────────────────────────────────────────────
# COLORMAPS AVAILABLE
# ─────────────────────────────────────────────────────────────────────────────
# Choose any of: 'jet', 'viridis', 'plasma', 'magma', 'inferno', 'turbo'
COLORMAP_CV2 = {
    'jet'     : cv2.COLORMAP_JET,
    'viridis' : cv2.COLORMAP_VIRIDIS,
    'plasma'  : cv2.COLORMAP_PLASMA,
    'magma'   : cv2.COLORMAP_MAGMA,
    'inferno'  : cv2.COLORMAP_INFERNO,
    'turbo'   : cv2.COLORMAP_TURBO,
}

# ─────────────────────────────────────────────────────────────────────────────
# PREDICT DEPTH
# ─────────────────────────────────────────────────────────────────────────────
def predict_depth(image_path):
    """Load image → run model → return (rgb_np [H,W,3], depth_m [H,W])"""
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_rgb.shape[:2]
    print(f"  Input image : {w_orig}x{h_orig}  dtype={img_rgb.dtype}")

    tensor = infer_tfm(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        depth_tensor = model(tensor.float())           # float32, no autocast

    raw = depth_tensor.squeeze().cpu().float().numpy() # (384,384) float32
    raw = np.ascontiguousarray(raw)
    print(f"  Raw output  : min={raw.min():.6f}  max={raw.max():.6f}  "
          f"mean={raw.mean():.6f}")

    # fallback: if model output is near-zero, normalise
    if raw.max() < 0.01:
        print("  ⚠ Near-zero output — applying min-max normalisation.")
        raw = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

    raw = raw.astype(np.float32)
    depth_resized = cv2.resize(raw, (w_orig, h_orig),
                               interpolation=cv2.INTER_LINEAR)
    depth_m = depth_resized * 10.0   # [0,1] → metres (NYU max = 10 m)

    print(f"  Depth (m)   : min={depth_m.min():.3f}  max={depth_m.max():.3f}")
    return img_rgb, depth_m


# ─────────────────────────────────────────────────────────────────────────────
# APPLY COLORMAP TO DEPTH → RGB image
# ─────────────────────────────────────────────────────────────────────────────
def depth_to_colormap(depth_m, colormap='turbo'):
    """
    Convert a float depth map [H,W] in metres to an RGB colormapped image [H,W,3].
    Near  = warm colours (red/yellow in jet/turbo)
    Far   = cool colours (blue/purple)
    """
    # Normalise to uint8 [0..255]
    d_min, d_max = depth_m.min(), depth_m.max()
    depth_norm = ((depth_m - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)

    cv2_cmap = COLORMAP_CV2.get(colormap, cv2.COLORMAP_TURBO)
    colored_bgr = cv2.applyColorMap(depth_norm, cv2_cmap)   # BGR
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    return colored_rgb   # (H, W, 3) uint8


# ─────────────────────────────────────────────────────────────────────────────
# SAVE COLORMAP DEPTH IMAGE + SIDE-BY-SIDE HTML
# ─────────────────────────────────────────────────────────────────────────────
def save_colormap_images(img_rgb, depth_m,
                         colormap='turbo',
                         out_dir='/kaggle/working'):
    """Save: colourmap PNG + side-by-side comparison PNG."""
    colored = depth_to_colormap(depth_m, colormap)

    cmap_path  = os.path.join(out_dir, f'depth_{colormap}.png')
    side_path  = os.path.join(out_dir, f'depth_{colormap}_sidebyside.png')

    # individual colourmap PNG
    cv2.imwrite(cmap_path, cv2.cvtColor(colored, cv2.COLOR_RGB2BGR))

    # side-by-side: RGB | colourmap
    H, W = img_rgb.shape[:2]
    gap = np.zeros((H, 20, 3), dtype=np.uint8) + 30   # dark separator
    side = np.hstack([img_rgb, gap, colored])
    cv2.imwrite(side_path, cv2.cvtColor(side, cv2.COLOR_RGB2BGR))

    print(f"  Saved colourmap PNG  : {cmap_path}")
    print(f"  Saved side-by-side   : {side_path}")
    return colored


# ─────────────────────────────────────────────────────────────────────────────
# BUILD 3D POINT CLOUD  (coloured by depth colormap OR original RGB)
# ─────────────────────────────────────────────────────────────────────────────
def build_pointcloud_html(image_path,
                          fx=518.8, fy=519.0,
                          cx=None,  cy=None,
                          step=4,
                          colormap='turbo',         # depth colormap for 3D cloud
                          use_rgb_for_cloud=False,  # True = original photo colours
                          save_path='/kaggle/working/pointcloud_measure.html'):
    """
    1. Run depth model
    2. Apply chosen colormap (turbo/jet/viridis/plasma/magma/inferno)
    3. Save side-by-side PNGs
    4. Build interactive 3D point cloud coloured by depth colormap
    5. Embed click-to-measure JS
    """

    print(f"\n{'─'*60}")
    print(f"Image     : {image_path}")
    print(f"Colormap  : {colormap}")
    print(f"{'─'*60}")

    img_rgb, depth_m = predict_depth(image_path)
    H, W = depth_m.shape

    if cx is None: cx = W / 2.0
    if cy is None: cy = H / 2.0

    # ── apply colormap ────────────────────────────────────────────────────────
    colored = save_colormap_images(img_rgb, depth_m, colormap,
                                   out_dir=os.path.dirname(save_path)
                                   or '/kaggle/working')

    # ── adaptive min_depth ────────────────────────────────────────────────────
    min_depth = 0.1
    valid_frac = np.mean(depth_m > min_depth)
    if valid_frac < 0.05:
        min_depth = float(np.percentile(depth_m, 5))
        print(f"  ⚠ Low valid fraction — lowering min_depth to {min_depth:.4f}m")

    print(f"  Depth range : {depth_m.min():.2f}m – {depth_m.max():.2f}m  |  "
          f"Valid pixels : {valid_frac*100:.1f}%")

    # ── choose colour source for 3D cloud ─────────────────────────────────────
    colour_src = img_rgb if use_rgb_for_cloud else colored  # (H,W,3)

    # ── back-projection ───────────────────────────────────────────────────────
    xs, ys, zs, colors, hover = [], [], [], [], []

    for v in range(0, H, step):
        for u in range(0, W, step):
            z = float(depth_m[v, u])
            if z < min_depth:
                continue
            x =  (u - cx) * z / fx
            y = -(v - cy) * z / fy
            r, g, b = colour_src[v, u]
            xs.append(round(x, 3))
            ys.append(round(y, 3))
            zs.append(round(-z, 3))
            colors.append(f'rgb({r},{g},{b})')
            hover.append(f"X: {x:.2f}m<br>Y: {y:.2f}m<br>"
                         f"Depth: {z:.2f}m<br>Pixel: ({u},{v})")

    print(f"  Total 3D points : {len(xs):,}")
    if len(xs) == 0:
        raise RuntimeError("No 3D points generated — check depth diagnostics above.")

    # ── plotly figure ─────────────────────────────────────────────────────────
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(        # trace 0 — point cloud
        x=xs, y=ys, z=zs,
        mode='markers',
        marker=dict(size=1.8, color=colors, opacity=0.95),
        text=hover,
        hovertemplate='%{text}<extra></extra>',
        name='Scene',
    ))
    fig.add_trace(go.Scatter3d(        # trace 1 — pins
        x=[], y=[], z=[],
        mode='markers+text',
        marker=dict(size=10, color=['#ff3333', '#33aaff'],
                    symbol='diamond',
                    line=dict(color='white', width=1.5)),
        text=[], textposition='top center',
        textfont=dict(color='white', size=13),
        name='Pins',
        hovertemplate='%{text}<extra></extra>',
    ))
    fig.add_trace(go.Scatter3d(        # trace 2 — distance line
        x=[], y=[], z=[],
        mode='lines',
        line=dict(color='yellow', width=5, dash='dash'),
        name='Distance',
        hoverinfo='skip',
    ))

    cmap_title = colormap.capitalize()
    fig.update_layout(
        title=dict(
            text=f'Monocular Depth — 3D Point Cloud [{cmap_title} colormap]'
                 '  |  Click any two points to measure',
            font=dict(size=14, color='white')
        ),
        paper_bgcolor='#0a0a0a',
        plot_bgcolor='#0a0a0a',
        scene=dict(
            bgcolor='#0a0a0a',
            xaxis=dict(title='X (m)', color='#aaa',
                       backgroundcolor='#111', gridcolor='#2a2a2a',
                       showbackground=True),
            yaxis=dict(title='Y (m)', color='#aaa',
                       backgroundcolor='#111', gridcolor='#2a2a2a',
                       showbackground=True),
            zaxis=dict(title='Depth (m)', color='#aaa',
                       backgroundcolor='#111', gridcolor='#2a2a2a',
                       showbackground=True),
            aspectmode='data',
            camera=dict(eye=dict(x=0, y=0, z=-2)),
        ),
        legend=dict(font=dict(color='white'), bgcolor='#1a1a1a',
                    bordercolor='#444', borderwidth=1),
        margin=dict(l=0, r=0, t=50, b=0),
        height=780,
    )

    # ── colormap legend bar (dummy heatmap for the colour scale) ──────────────
    fig.add_trace(go.Scatter3d(        # trace 3 — invisible, just for colorbar
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(
            size=0,
            color=[depth_m.min(), depth_m.max()],
            colorscale=colormap,
            showscale=True,
            colorbar=dict(
                title=dict(text='Depth (m)', font=dict(color='white', size=12)),
                tickfont=dict(color='white'),
                thickness=15,
                len=0.6,
                x=1.02,
            ),
        ),
        showlegend=False,
        hoverinfo='skip',
    ))

    # ── JavaScript click-to-measure ───────────────────────────────────────────
    js = r"""
<script>
(function() {
    function waitForPlotly(cb) {
        var el = document.querySelector('.plotly-graph-div');
        if (el && el._fullLayout) { cb(el); return; }
        setTimeout(function(){ waitForPlotly(cb); }, 300);
    }
    waitForPlotly(function(gd) {
        var pins = [], history = [];

        gd.on('plotly_click', function(data) {
            var pt = data.points[0];
            if (pt.curveNumber !== 0) return;
            var p = { x: pt.x, y: pt.y, z: pt.z };
            pins.push(p);

            var labels = pins.map(function(_, i){ return 'P'+(i+1); });
            var pinColors = pins.map(function(_, i){
                return i === 0 ? '#ff4444' : '#44aaff';
            });
            Plotly.restyle(gd, {
                x: [pins.map(function(p){ return p.x; })],
                y: [pins.map(function(p){ return p.y; })],
                z: [pins.map(function(p){ return p.z; })],
                text: [labels],
                'marker.color': [pinColors],
            }, 1);

            setStatus('<span style="color:#4af">P'+pins.length+
                      ' set at ('+p.x.toFixed(2)+', '+
                      p.y.toFixed(2)+', '+p.z.toFixed(2)+' m)</span>'+
                      (pins.length===1 ? '<br>Now click a second point...' : ''));

            if (pins.length === 2) {
                var p1=pins[0], p2=pins[1];
                var d = Math.sqrt(
                    Math.pow(p2.x-p1.x,2)+
                    Math.pow(p2.y-p1.y,2)+
                    Math.pow(p2.z-p1.z,2)
                ).toFixed(3);

                Plotly.restyle(gd,{
                    x:[[p1.x,p2.x]], y:[[p1.y,p2.y]], z:[[p1.z,p2.z]]
                },2);

                history.push({n:history.length+1,
                              p1:fmtPt(p1),p2:fmtPt(p2),dist:d});
                setStatus(
                    '<span style="color:yellow;font-size:16px;font-weight:bold">'+
                    'Distance: '+d+' m</span><br>'+
                    '<span style="color:#bbb;font-size:12px">'+
                    'P1 '+fmtPt(p1)+'  →  P2 '+fmtPt(p2)+'</span>'
                );
                updateTable();
                setTimeout(function(){
                    pins=[];
                    Plotly.restyle(gd,{x:[[]],y:[[]],z:[[]]},1);
                    Plotly.restyle(gd,{x:[[]],y:[[]],z:[[]]},2);
                    setStatus('<span style="color:#777">Click a point to start measuring</span>');
                },3000);
            }
        });

        function fmtPt(p){
            return '('+p.x.toFixed(2)+', '+p.y.toFixed(2)+', '+p.z.toFixed(2)+')';
        }
        function setStatus(html){
            var el=document.getElementById('status-box');
            if(el) el.innerHTML=html;
        }
        function updateTable(){
            var tbody=document.getElementById('hist-body');
            if(!tbody) return;
            tbody.innerHTML=history.slice().reverse().map(function(r){
                return '<tr style="border-bottom:1px solid #2a2a2a">'+
                    '<td style="padding:4px 6px;color:#888">'+r.n+'</td>'+
                    '<td style="padding:4px 6px;color:#ccc">'+r.p1+'</td>'+
                    '<td style="padding:4px 6px;color:#ccc">'+r.p2+'</td>'+
                    '<td style="padding:4px 6px;color:yellow;font-weight:bold">'+r.dist+' m</td>'+
                    '</tr>';
            }).join('');
        }
        window._clearMeasure=function(){
            pins=[];history=[];
            Plotly.restyle(gd,{x:[[]],y:[[]],z:[[]]},[ 1,2]);
            setStatus('<span style="color:#777">Click a point to start measuring</span>');
            var tbody=document.getElementById('hist-body');
            if(tbody) tbody.innerHTML='';
        };
        window._exportCSV=function(){
            if(history.length===0){alert('No measurements yet.');return;}
            var csv='#,P1 (x y z m),P2 (x y z m),Distance (m)\n';
            history.forEach(function(r){
                csv+=r.n+','+r.p1+','+r.p2+','+r.dist+'\n';
            });
            var a=document.createElement('a');
            a.href='data:text/csv;charset=utf-8,'+encodeURIComponent(csv);
            a.download='measurements.csv';
            a.click();
        };
    });
})();
</script>

<!-- ── Control panel ─────────────────────────────────────────────── -->
<div style="margin:12px auto;max-width:900px;background:#111;border:1px solid #333;
            border-radius:12px;padding:16px;font-family:monospace;color:white;">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px">
    <div>
      <span style="font-size:15px;font-weight:bold;color:#4af">Distance Ruler</span>
      <span id="cmap-badge" style="margin-left:10px;font-size:11px;
            background:#1a3a1a;color:#4f4;border:1px solid #2a6a2a;
            border-radius:4px;padding:2px 8px;">COLORMAP_LABEL</span>
    </div>
    <div style="display:flex;gap:8px">
      <button onclick="_clearMeasure()" style="background:#222;color:#ccc;border:1px solid #444;
              border-radius:6px;padding:5px 14px;cursor:pointer;font-size:13px">Clear</button>
      <button onclick="_exportCSV()" style="background:#0a2a0a;color:#4f4;border:1px solid #1a5a1a;
              border-radius:6px;padding:5px 14px;cursor:pointer;font-size:13px">Export CSV</button>
    </div>
  </div>
  <div id="status-box" style="background:#0d0d0d;border-radius:8px;padding:12px;
       min-height:48px;margin-bottom:12px;font-size:13px;line-height:1.6">
    <span style="color:#777">Click a point to start measuring</span>
  </div>
  <div style="max-height:160px;overflow-y:auto">
    <table style="width:100%;font-size:11px;border-collapse:collapse">
      <thead>
        <tr style="color:#666;border-bottom:1px solid #333">
          <th style="padding:4px 6px;text-align:left">#</th>
          <th style="padding:4px 6px;text-align:left">P1 (x, y, z)</th>
          <th style="padding:4px 6px;text-align:left">P2 (x, y, z)</th>
          <th style="padding:4px 6px;text-align:left">Distance</th>
        </tr>
      </thead>
      <tbody id="hist-body"></tbody>
    </table>
  </div>
  <div style="margin-top:10px;font-size:10px;color:#555">
    🔴 Red diamond = P1 &nbsp;|&nbsp; 🔵 Blue diamond = P2 &nbsp;|&nbsp;
    🟡 Yellow line = distance &nbsp;|&nbsp; Auto-resets after 3s
  </div>
</div>
"""
    js = js.replace('COLORMAP_LABEL', colormap.upper())

    # ── write HTML ────────────────────────────────────────────────────────────
    html = fig.to_html(full_html=True, include_plotlyjs=True)
    html = html.replace('</body>', js + '</body>')

    with open(save_path, 'w') as f:
        f.write(html)

    size_mb = os.path.getsize(save_path) / 1e6
    print(f"\n✅ Saved → {save_path}  ({size_mb:.1f} MB)")
    print("─" * 60)
    print("HOW TO USE:")
    print("  1. Download and open in Chrome or Firefox")
    print("  2. Rotate  → click + drag   |   Zoom → scroll wheel")
    print("  3. Measure → click P1 (red diamond) → click P2 (blue diamond)")
    print("             → yellow dashed line + distance in metres")
    print("  5. Export  → 'Export CSV' button")
    print("─" * 60)
    return save_path


# ─────────────────────────────────────────────────────────────────────────────
# DISCOVER TEST IMAGES
# ─────────────────────────────────────────────────────────────────────────────
TEST_DIR    = "/kaggle/input/nyu-depth-v2/nyu_data/data/nyu2_test"
test_images = sorted(glob.glob(os.path.join(TEST_DIR, "*_colors.png")))

if not test_images:
    raise FileNotFoundError(
        f"No '*_colors.png' files found in {TEST_DIR}.\n"
        "Check the dataset path or file naming."
    )

print(f"Found {len(test_images)} test colour images.")
print("First 5:", test_images[:5])

# ─────────────────────────────────────────────────────────────────────────────
# RUN
# Change colormap to: 'jet' | 'viridis' | 'plasma' | 'magma' | 'inferno' | 'turbo'
# ─────────────────────────────────────────────────────────────────────────────
IMAGE_PATH = test_images[0]

build_pointcloud_html(
    image_path        = IMAGE_PATH,
    fx                = 518.8,
    fy                = 519.0,
    step              = 4,
    colormap          = 'turbo',   # ← change colormap here
    use_rgb_for_cloud = False,     # False = colormap colours | True = original photo
    save_path         = '/kaggle/working/pointcloud_measure.html'
)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          MONOCULAR DEPTH ESTIMATION — INTERACTIVE ANALYSIS DASHBOARD        ║
║          EfficientNet-B5 + UNet++ with SCSE Attention | NYU Depth V2        ║
║          Author  : [Your Name]                                               ║
║          Model   : UNetPlusPlus | Encoder: EfficientNet-B5 | Dataset: NYUv2 ║
╚══════════════════════════════════════════════════════════════════════════════╝

NOVEL FEATURES (beyond baseline):
  ① Auto-adaptive depth normalisation  — handles near-zero / saturated outputs
  ② Perspective-correct back-projection with focal-length intrinsics
  ③ Real-time 3-D distance simulator   — click anchor → 5-NN lines + table
  ④ Depth-gradient edge-saliency map   — highlights depth discontinuities
  ⑤ Depth histogram + cumulative curve — per-image depth distribution analysis
  ⑥ Surface-normal estimation          — derived from depth gradients
  ⑦ Depth cross-section profiler       — horizontal / vertical depth slices
  ⑧ Multi-colormap side-by-side panel  — Turbo / Jet / Viridis with live axes
  ⑨ Per-region depth statistics table  — 3×3 spatial grid statistics
  ⑩ Fully inline Kaggle rendering      — no external files needed
"""

# ──────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
import cv2
import glob
import os
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from IPython.display import display, HTML

# ──────────────────────────────────────────────────────────────────────────────
# DEVICE
# ──────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{'='*65}")
print(f"  🔧  Device : {DEVICE}")
print(f"{'='*65}")

# ──────────────────────────────────────────────────────────────────────────────
# MODEL DEFINITION
# ──────────────────────────────────────────────────────────────────────────────
ENCODER     = 'efficientnet-b5'
ENCODER_WTS = 'imagenet'

class DepthModel(nn.Module):
    """
    UNet++ with EfficientNet-B5 encoder, SCSE decoder attention.
    Output: sigmoid-normalised depth map in [0, 1].
    """
    def __init__(self, encoder=ENCODER, weights=ENCODER_WTS):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name        = encoder,
            encoder_weights     = weights,
            in_channels         = 3,
            classes             = 1,
            activation          = None,
            decoder_attention_type = 'scse',
        )

    def forward(self, x):
        return torch.sigmoid(self.model(x))


# ──────────────────────────────────────────────────────────────────────────────
# LOAD CHECKPOINT
# ──────────────────────────────────────────────────────────────────────────────
PTH_PATH = "/kaggle/working/depth_efficientnet_b5_unetplusplus.pt"

model = DepthModel().to(DEVICE)
ckpt  = torch.load(PTH_PATH, map_location=DEVICE)

if 'state_dict' in ckpt:
    model.load_state_dict(ckpt['state_dict'])
    print(f"  ✅  Checkpoint loaded  —  epoch {ckpt.get('epoch','?')} "
          f"| SSIM {ckpt.get('ssim', 0):.4f} | RMSE {ckpt.get('rmse', 0):.4f}")
else:
    model.load_state_dict(ckpt)
    print("  ✅  Raw state_dict loaded")

model.eval()
print(f"{'='*65}\n")

# ──────────────────────────────────────────────────────────────────────────────
# INFERENCE TRANSFORMS  (ImageNet normalisation → 384×384)
# ──────────────────────────────────────────────────────────────────────────────
IMG_SIZE = 384
MEAN     = (0.485, 0.456, 0.406)
STD      = (0.229, 0.224, 0.225)

infer_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# ──────────────────────────────────────────────────────────────────────────────
# PREDICT DEPTH  (fix 1: always normalise; fix 2: print raw stats)
# ──────────────────────────────────────────────────────────────────────────────
def predict_depth(image_path: str, max_depth_m: float = 10.0):
    """
    Returns
    -------
    img_rgb   : H×W×3  uint8
    depth_m   : H×W    float32  values in metres (0 … max_depth_m)
    raw_norm  : H×W    float32  raw sigmoid output normalised to [0,1]
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]

    tensor = infer_tfm(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(tensor.float())

    raw = out.squeeze().cpu().float().numpy()
    raw = np.ascontiguousarray(raw)

    # ── diagnostic ──────────────────────────────────────────────
    print(f"  Raw sigmoid  →  min={raw.min():.5f}  max={raw.max():.5f}"
          f"  mean={raw.mean():.5f}  std={raw.std():.5f}")

    # ── FIX: always stretch to full [0,1] range ──────────────────
    # Prevents narrow-range outputs from collapsing the colormap
    raw_norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

    # ── resize back to original resolution ──────────────────────
    raw_resized = cv2.resize(raw_norm.astype(np.float32), (W, H),
                             interpolation=cv2.INTER_LINEAR)

    # ── scale to metres ──────────────────────────────────────────
    depth_m = raw_resized * max_depth_m

    print(f"  Depth (m)    →  min={depth_m.min():.3f}  max={depth_m.max():.3f}"
          f"  mean={depth_m.mean():.3f}  std={depth_m.std():.3f}")

    return img_rgb, depth_m, raw_norm


# ──────────────────────────────────────────────────────────────────────────────
# COLORMAP HELPER
# ──────────────────────────────────────────────────────────────────────────────
CV2_CMAPS = {
    'turbo'  : cv2.COLORMAP_TURBO,
    'jet'    : cv2.COLORMAP_JET,
    'viridis': cv2.COLORMAP_VIRIDIS,
    'plasma' : cv2.COLORMAP_PLASMA,
    'magma'  : cv2.COLORMAP_MAGMA,
    'inferno': cv2.COLORMAP_INFERNO,
}
PLOTLY_CMAPS = {
    'turbo'  : 'Turbo',  'jet'    : 'Jet',
    'viridis': 'Viridis','plasma' : 'Plasma',
    'magma'  : 'Magma',  'inferno': 'Inferno',
}

def depth_to_rgb(depth_m: np.ndarray, cmap: str = 'turbo') -> np.ndarray:
    """Convert depth map to false-colour RGB image."""
    d_min, d_max = depth_m.min(), depth_m.max()
    norm = ((depth_m - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)
    bgr  = cv2.applyColorMap(norm, CV2_CMAPS.get(cmap, cv2.COLORMAP_TURBO))
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


# ──────────────────────────────────────────────────────────────────────────────
# NOVEL FEATURE ④ — DEPTH-GRADIENT EDGE-SALIENCY MAP
# ──────────────────────────────────────────────────────────────────────────────
def compute_edge_saliency(depth_m: np.ndarray) -> np.ndarray:
    """
    Sobel gradient magnitude on the depth map highlights
    depth discontinuities (object boundaries).
    Returns float32 image normalised to [0,1].
    """
    norm = ((depth_m - depth_m.min()) /
            (depth_m.max() - depth_m.min() + 1e-8) * 255).astype(np.uint8)
    gx   = cv2.Sobel(norm, cv2.CV_64F, 1, 0, ksize=3)
    gy   = cv2.Sobel(norm, cv2.CV_64F, 0, 1, ksize=3)
    mag  = np.sqrt(gx**2 + gy**2)
    return (mag / (mag.max() + 1e-8)).astype(np.float32)


# ──────────────────────────────────────────────────────────────────────────────
# NOVEL FEATURE ⑥ — SURFACE-NORMAL ESTIMATION FROM DEPTH GRADIENTS
# ──────────────────────────────────────────────────────────────────────────────
def compute_surface_normals(depth_m: np.ndarray,
                             fx: float = 518.8,
                             fy: float = 519.0) -> np.ndarray:
    """
    Estimate per-pixel surface normals from depth gradients.
    Returns H×W×3 float32 array (nx, ny, nz) each in [-1,1].
    """
    H, W  = depth_m.shape
    dzdx  = cv2.Sobel(depth_m, cv2.CV_64F, 1, 0, ksize=5)
    dzdy  = cv2.Sobel(depth_m, cv2.CV_64F, 0, 1, ksize=5)

    # 3-D tangent vectors in camera space
    tx = np.stack([np.ones((H, W)) / fx,
                   np.zeros((H, W)),
                   dzdx], axis=-1)
    ty = np.stack([np.zeros((H, W)),
                   np.ones((H, W)) / fy,
                   dzdy], axis=-1)

    # normal = tx × ty
    nx = ty[..., 1] * tx[..., 2] - ty[..., 2] * tx[..., 1]
    ny = ty[..., 2] * tx[..., 0] - ty[..., 0] * tx[..., 2]
    nz = ty[..., 0] * tx[..., 1] - ty[..., 1] * tx[..., 0]

    norm  = np.sqrt(nx**2 + ny**2 + nz**2) + 1e-8
    normals = np.stack([nx / norm, ny / norm, nz / norm], axis=-1)
    return normals.astype(np.float32)


def normals_to_rgb(normals: np.ndarray) -> np.ndarray:
    """Map [-1,1] normals to [0,255] RGB for visualisation."""
    return ((normals * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)


# ──────────────────────────────────────────────────────────────────────────────
# NOVEL FEATURE ⑨ — PER-REGION DEPTH STATISTICS  (3×3 spatial grid)
# ──────────────────────────────────────────────────────────────────────────────
def compute_region_stats(depth_m: np.ndarray) -> list:
    """
    Divides the depth map into a 3×3 spatial grid and returns
    per-cell statistics: mean, median, std, min, max depth (m).
    """
    H, W = depth_m.shape
    rows_  = np.array_split(np.arange(H), 3)
    cols_  = np.array_split(np.arange(W), 3)
    labels = ['Top-L', 'Top-C', 'Top-R',
              'Mid-L', 'Mid-C', 'Mid-R',
              'Bot-L', 'Bot-C', 'Bot-R']
    stats  = []
    for ri, row_idx in enumerate(rows_):
        for ci, col_idx in enumerate(cols_):
            patch = depth_m[np.ix_(row_idx, col_idx)]
            stats.append({
                'region': labels[ri*3 + ci],
                'mean'  : float(patch.mean()),
                'median': float(np.median(patch)),
                'std'   : float(patch.std()),
                'min'   : float(patch.min()),
                'max'   : float(patch.max()),
            })
    return stats


# ──────────────────────────────────────────────────────────────────────────────
# BACK-PROJECTION  (fix: auto min_depth; vectorised for speed)
# ──────────────────────────────────────────────────────────────────────────────
def backproject(depth_m: np.ndarray,
                colour_rgb: np.ndarray,
                fx: float = 518.8, fy: float = 519.0,
                cx: float = None,  cy: float = None,
                step: int = 4,     min_depth: float = 0.3) -> dict:
    """
    Perspective back-projection: pixel (u,v) + depth → 3-D point (X,Y,Z).
    Vectorised — no Python loop.
    """
    H, W = depth_m.shape
    if cx is None: cx = W / 2.0
    if cy is None: cy = H / 2.0

    # ── auto-adjust min_depth to guarantee >5 % valid pixels ──
    p5  = float(np.percentile(depth_m, 5))
    p95 = float(np.percentile(depth_m, 95))
    effective_min = max(min_depth, p5 * 0.5)

    print(f"    Backproject: p5={p5:.2f}m  p95={p95:.2f}m  "
          f"effective_min={effective_min:.2f}m")

    # ── build pixel grids ──────────────────────────────────────
    us = np.arange(0, W, step)
    vs = np.arange(0, H, step)
    uu, vv = np.meshgrid(us, vs)
    uu = uu.ravel(); vv = vv.ravel()

    zz = depth_m[vv, uu]
    mask = zz > effective_min
    uu, vv, zz = uu[mask], vv[mask], zz[mask]

    xx =  (uu - cx) * zz / fx
    yy = -(vv - cy) * zz / fy

    r = colour_rgb[vv, uu, 0]
    g = colour_rgb[vv, uu, 1]
    b = colour_rgb[vv, uu, 2]
    cols    = [f'rgb({ri},{gi},{bi})' for ri, gi, bi in zip(r, g, b)]
    hovers  = [f'X:{xi:.2f}m  Y:{yi:.2f}m  D:{zi:.2f}m  px:({ui},{vi})'
               for xi, yi, zi, ui, vi in
               zip(xx.round(2), yy.round(2), zz.round(2), uu, vv)]

    return dict(x=xx.round(3).tolist(),
                y=yy.round(3).tolist(),
                z=(-zz).round(3).tolist(),
                color=cols,
                hover=hovers)


# ──────────────────────────────────────────────────────────────────────────────
# COLOUR THEME
# ──────────────────────────────────────────────────────────────────────────────
BG   = '#060810'
BG2  = '#0d1117'
GRID = '#1e2433'
TEXT = '#c9d1d9'
ACC1 = '#58a6ff'
ACC2 = '#7ee787'
ACC3 = '#f78166'
ACC4 = '#d2a8ff'

_layout_base = dict(
    paper_bgcolor = BG,
    plot_bgcolor  = BG2,
    font          = dict(color=TEXT, family='monospace', size=11),
)
_margin_default = dict(l=50, r=30, t=55, b=40)
_margin_3d      = dict(l=0,  r=0,  t=55, b=0)

_axis_kw = dict(
    showgrid   = True,
    gridcolor  = GRID,
    gridwidth  = 0.5,
    zeroline   = False,
    tickfont   = dict(color='#8b949e', size=9),
    title_font = dict(color='#8b949e'),
)

_scene_kw = dict(
    bgcolor         = BG,
    xaxis           = dict(title='X (m)', color='#8b949e',
                           backgroundcolor=BG2, gridcolor=GRID,
                           showbackground=True, tickfont=dict(size=9)),
    yaxis           = dict(title='Y (m)', color='#8b949e',
                           backgroundcolor=BG2, gridcolor=GRID,
                           showbackground=True, tickfont=dict(size=9)),
    zaxis           = dict(title='Depth (m)', color='#8b949e',
                           backgroundcolor=BG2, gridcolor=GRID,
                           showbackground=True, tickfont=dict(size=9)),
    aspectmode      = 'data',
    camera          = dict(eye=dict(x=0.0, y=-0.3, z=-1.8)),
)


# ══════════════════════════════════════════════════════════════════════════════
# PANEL A — ORIGINAL RGB + 3 DEPTH COLORMAPS
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_a(img_rgb, depth_m, colormaps, d_min, d_max):
    titles = ['Original RGB'] + [f'Depth: {c.capitalize()}' for c in colormaps]
    fig    = make_subplots(rows=1, cols=4,
                           subplot_titles=titles,
                           horizontal_spacing=0.03)

    fig.add_trace(go.Image(z=img_rgb, name='RGB'), row=1, col=1)

    for i, c in enumerate(colormaps, start=2):
        show_cb = (i == 4)
        fig.add_trace(go.Heatmap(
            z            = depth_m,
            colorscale   = PLOTLY_CMAPS.get(c, 'Turbo'),
            zmin         = d_min, zmax = d_max,
            colorbar     = dict(
                title    = dict(text='depth (m)', font=dict(color=TEXT, size=10)),
                tickfont = dict(color='#8b949e', size=9),
                len=0.90, thickness=12,
                x=1.01,
                bgcolor  = BG2,
                bordercolor = GRID,
            ) if show_cb else dict(showticklabels=False, len=0, thickness=0),
            showscale        = show_cb,
            name             = c,
            hovertemplate    = 'col=%{x}  row=%{y}<br>depth=%{z:.3f} m<extra></extra>',
        ), row=1, col=i)

    fig.update_layout(
        **_layout_base,
        margin = _margin_default,
        height = 380,
        title  = dict(
            text = '🎨  Panel A — Predicted Depth Maps with 3 Colormaps',
            font = dict(size=13, color=ACC1),
        ),
    )
    for col_i in range(1, 5):
        fig.update_xaxes(title_text='pixel X', **_axis_kw, row=1, col=col_i)
        fig.update_yaxes(title_text='pixel Y', autorange='reversed',
                         **_axis_kw, row=1, col=col_i)

    for ann in fig.layout.annotations:
        ann.font.color  = ACC4
        ann.font.size   = 11

    return fig


# ══════════════════════════════════════════════════════════════════════════════
# PANEL B — NOVEL: EDGE SALIENCY + SURFACE NORMALS + DEPTH DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_b(img_rgb, depth_m, edge_sal, normals_rgb):
    H, W = depth_m.shape

    fig = make_subplots(
        rows=1, cols=4,
        subplot_titles=[
            'Edge Saliency (depth discontinuities)',
            'Surface Normals (from ∂depth)',
            'Depth Histogram',
            'Cumulative Depth Distribution',
        ],
        horizontal_spacing=0.06,
    )

    # ── Edge Saliency ─────────────────────────────────────────────────────
    edge_u8 = (edge_sal * 255).astype(np.uint8)
    edge_rgb = cv2.cvtColor(
        cv2.applyColorMap(edge_u8, cv2.COLORMAP_INFERNO),
        cv2.COLOR_BGR2RGB)
    fig.add_trace(go.Image(z=edge_rgb, name='Edge Saliency'), row=1, col=1)

    # ── Surface Normals ───────────────────────────────────────────────────
    fig.add_trace(go.Image(z=normals_rgb, name='Normals'), row=1, col=2)

    # ── Depth Histogram ───────────────────────────────────────────────────
    flat = depth_m.ravel()
    counts, edges = np.histogram(flat, bins=60)
    bin_centres   = (edges[:-1] + edges[1:]) / 2.0

    fig.add_trace(go.Bar(
        x            = bin_centres.round(3).tolist(),
        y            = counts.tolist(),
        marker_color = ACC1,
        marker_line_width = 0,
        name         = 'Histogram',
        hovertemplate= 'depth=%{x:.2f}m<br>count=%{y}<extra></extra>',
    ), row=1, col=3)

    # ── Cumulative Distribution ────────────────────────────────────────────
    sorted_d = np.sort(flat)
    cdf      = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    # downsample for rendering speed
    idx  = np.linspace(0, len(sorted_d)-1, 500, dtype=int)
    fig.add_trace(go.Scatter(
        x    = sorted_d[idx].round(3).tolist(),
        y    = cdf[idx].round(4).tolist(),
        mode = 'lines',
        line = dict(color=ACC2, width=2),
        name = 'CDF',
        fill = 'tozeroy',
        fillcolor = 'rgba(126,231,135,0.08)',
        hovertemplate = 'depth=%{x:.2f}m<br>CDF=%{y:.3f}<extra></extra>',
    ), row=1, col=4)

    # ── median line ───────────────────────────────────────────────────────
    med = float(np.median(flat))
    fig.add_vline(x=med, line_color=ACC3, line_dash='dash',
                  line_width=1.5, row=1, col=3)
    fig.add_vline(x=med, line_color=ACC3, line_dash='dash',
                  line_width=1.5, row=1, col=4)
    fig.add_annotation(x=med, y=counts.max()*0.9,
                       text=f'median<br>{med:.2f}m',
                       showarrow=False, font=dict(color=ACC3, size=9),
                       row=1, col=3)

    fig.update_layout(
        **_layout_base,
        margin     = _margin_default,
        height     = 340,
        showlegend = False,
        title      = dict(
            text = '🔬  Panel B — Depth Saliency · Surface Normals · Distribution Analysis',
            font = dict(size=13, color=ACC1),
        ),
    )
    for col_i in [3, 4]:
        fig.update_xaxes(title_text='depth (m)', **_axis_kw, row=1, col=col_i)
    fig.update_yaxes(title_text='count',  **_axis_kw, row=1, col=3)
    fig.update_yaxes(title_text='CDF',    **_axis_kw, row=1, col=4)
    for ann in fig.layout.annotations:
        ann.font.color = ACC4
        ann.font.size  = 11

    return fig


# ══════════════════════════════════════════════════════════════════════════════
# PANEL C — NOVEL: DEPTH CROSS-SECTION PROFILER  (horizontal + vertical)
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_c(img_rgb, depth_m):
    H, W = depth_m.shape
    row_frac = [0.25, 0.50, 0.75]
    col_frac = [0.25, 0.50, 0.75]

    colours_h = [ACC1, ACC2, ACC3]
    colours_v = [ACC4, '#ffa657', '#79c0ff']

    fig = make_subplots(
        rows=1, cols=3,
        column_widths=[0.45, 0.275, 0.275],
        subplot_titles=[
            'Cross-section scan lines (on depth map)',
            'Horizontal depth profiles',
            'Vertical depth profiles',
        ],
        horizontal_spacing=0.06,
    )

    # ── depth map with scan lines overlaid ──────────────────────────────
    depth_img = depth_to_rgb(depth_m, 'turbo')
    fig.add_trace(go.Image(z=depth_img, name='depth'), row=1, col=1)

    x_vals = np.arange(W)
    y_vals = np.arange(H)

    for frac, col in zip(row_frac, colours_h):
        r = int(frac * H)
        fig.add_shape(type='line', x0=0, x1=W-1, y0=r, y1=r,
                      line=dict(color=col, width=1.5, dash='dot'),
                      row=1, col=1)

    for frac, col in zip(col_frac, colours_v):
        c = int(frac * W)
        fig.add_shape(type='line', x0=c, x1=c, y0=0, y1=H-1,
                      line=dict(color=col, width=1.5, dash='dot'),
                      row=1, col=1)

    # ── horizontal profiles ───────────────────────────────────────────────
    for frac, col in zip(row_frac, colours_h):
        r    = int(frac * H)
        prof = depth_m[r, :]
        fig.add_trace(go.Scatter(
            x    = x_vals.tolist(),
            y    = prof.round(3).tolist(),
            mode = 'lines',
            line = dict(color=col, width=1.5),
            name = f'H @ {int(frac*100)}%',
            hovertemplate = 'col=%{x}<br>depth=%{y:.3f}m<extra></extra>',
        ), row=1, col=2)

    # ── vertical profiles ─────────────────────────────────────────────────
    for frac, col in zip(col_frac, colours_v):
        c    = int(frac * W)
        prof = depth_m[:, c]
        fig.add_trace(go.Scatter(
            x    = prof.round(3).tolist(),
            y    = y_vals.tolist(),
            mode = 'lines',
            line = dict(color=col, width=1.5),
            name = f'V @ {int(frac*100)}%',
            hovertemplate = 'depth=%{x:.3f}m<br>row=%{y}<extra></extra>',
        ), row=1, col=3)

    fig.update_layout(
        **_layout_base,
        margin     = _margin_default,
        height     = 360,
        showlegend = True,
        legend     = dict(font=dict(size=9, color=TEXT),
                          bgcolor=BG2, bordercolor=GRID, borderwidth=1,
                          x=0.34, y=0.98, xanchor='left'),
        title      = dict(
            text = '📐  Panel C — Depth Cross-Section Profiler (Horizontal & Vertical)',
            font = dict(size=13, color=ACC1),
        ),
    )
    fig.update_xaxes(title_text='pixel X',  **_axis_kw, row=1, col=2)
    fig.update_yaxes(title_text='depth (m)',**_axis_kw, row=1, col=2)
    fig.update_xaxes(title_text='depth (m)',**_axis_kw, row=1, col=3)
    fig.update_yaxes(title_text='pixel Y', autorange='reversed',
                     **_axis_kw, row=1, col=3)
    for ann in fig.layout.annotations:
        ann.font.color = ACC4
        ann.font.size  = 11
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# PANEL D — 3 INTERACTIVE 3-D POINT CLOUDS
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_d(clouds, colormaps):
    fig = make_subplots(
        rows=1, cols=3,
        specs            = [[{'type':'scatter3d'}]*3],
        subplot_titles   = [f'3D Cloud: {c.capitalize()}' for c in colormaps],
        horizontal_spacing = 0.01,
    )
    for i, c in enumerate(colormaps, start=1):
        cl = clouds[c]
        fig.add_trace(go.Scatter3d(
            x    = cl['x'], y = cl['y'], z = cl['z'],
            mode = 'markers',
            marker = dict(size=1.4, color=cl['color'], opacity=0.9),
            text = cl['hover'],
            hovertemplate = '%{text}<extra></extra>',
            name = c,
        ), row=1, col=i)

    fig.update_layout(
        **_layout_base,
        height     = 520,
        showlegend = False,
        title      = dict(
            text = '🌐  Panel D — Interactive 3-D Point Clouds  (drag to rotate · scroll to zoom · hover for coords)',
            font = dict(size=13, color=ACC1),
        ),
        margin  = _margin_3d,
        scene   = _scene_kw,
        scene2  = _scene_kw,
        scene3  = _scene_kw,
    )
    for ann in fig.layout.annotations:
        ann.font.color = ACC4; ann.font.size = 11
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# PANEL E — NOVEL: PER-REGION STATISTICS HEATMAP  (3×3 grid)
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_e(depth_m, region_stats):
    # Build 3×3 arrays for each metric
    metrics = ['mean', 'median', 'std']
    titles  = ['Mean depth (m)', 'Median depth (m)', 'Std dev (m)']

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=titles,
                        horizontal_spacing=0.10)

    region_labels = [s['region'] for s in region_stats]
    row_names = ['Top', 'Mid', 'Bot']
    col_names = ['Left', 'Centre', 'Right']

    for col_i, (metric, title) in enumerate(zip(metrics, titles), start=1):
        vals = np.array([s[metric] for s in region_stats]).reshape(3, 3)
        fig.add_trace(go.Heatmap(
            z             = vals.round(3).tolist(),
            x             = col_names,
            y             = row_names,
            colorscale    = 'Turbo',
            showscale     = (col_i == 3),
            text          = [[f'{v:.2f}m' for v in row] for row in vals.tolist()],
            texttemplate  = '%{text}',
            textfont      = dict(size=13, color='white'),
            hovertemplate = 'region=%{y}-%{x}<br>' + metric + '=%{z:.3f}m<extra></extra>',
            name          = metric,
        ), row=1, col=col_i)

    fig.update_layout(
        **_layout_base,
        margin = _margin_default,
        height = 280,
        title  = dict(
            text = '📊  Panel E — Per-Region Depth Statistics (3×3 Spatial Grid)',
            font = dict(size=13, color=ACC1),
        ),
    )
    for ann in fig.layout.annotations:
        ann.font.color = ACC4; ann.font.size = 11
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# PANEL F — NOVEL: DISTANCE SIMULATOR  (3-D, click anchor → 5-NN lines)
# ══════════════════════════════════════════════════════════════════════════════
def build_panel_f(clouds, colormaps):
    sim_cloud = clouds[colormaps[0]]
    sim_x  = np.array(sim_cloud['x'])
    sim_y  = np.array(sim_cloud['y'])
    sim_z  = np.array(sim_cloud['z'])
    sim_col = sim_cloud['color']

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=sim_x.tolist(), y=sim_y.tolist(), z=sim_z.tolist(),
        mode='markers',
        marker=dict(size=1.6, color=sim_col, opacity=0.88),
        text=sim_cloud['hover'],
        hovertemplate='%{text}<extra></extra>',
        name='Scene',
        customdata=np.stack([sim_x, sim_y, sim_z], axis=-1).tolist(),
    ))

    fig.add_trace(go.Scatter3d(
        x=[], y=[], z=[],
        mode='markers+text',
        marker=dict(size=14, color=ACC3, symbol='diamond',
                    line=dict(color='white', width=2)),
        text=['ANCHOR'], textposition='top center',
        textfont=dict(color='white', size=12),
        name='Anchor',
        hovertemplate='%{text}<extra></extra>',
    ))

    nn_colors = [ACC3, '#ffd93d', ACC2, ACC1, ACC4]
    for k in range(5):
        fig.add_trace(go.Scatter3d(
            x=[], y=[], z=[],
            mode='lines+text',
            line=dict(color=nn_colors[k], width=4),
            text=[], textposition='middle center',
            textfont=dict(color='white', size=10),
            name=f'NN-{k+1}',
            hoverinfo='skip',
        ))

    fig.update_layout(
        **_layout_base,
        height = 580,
        title  = dict(
            text = '🎯  Panel F — 3D Distance Simulator  (click any point → red anchor → 5 nearest neighbours)',
            font = dict(size=13, color=ACC1),
        ),
        scene  = _scene_kw,
        legend = dict(font=dict(color=TEXT, size=10), bgcolor=BG2,
                      bordercolor=GRID, borderwidth=1),
        margin = _margin_3d,
    )
    return fig, sim_x, sim_y, sim_z


# ══════════════════════════════════════════════════════════════════════════════
# SIMULATOR JAVASCRIPT  (injected after Panel F HTML)
# ══════════════════════════════════════════════════════════════════════════════
def build_sim_js(sim_x, sim_y, sim_z):
    pts_js = (
        f'var PTS_X={sim_x.round(3).tolist()};\n'
        f'var PTS_Y={sim_y.round(3).tolist()};\n'
        f'var PTS_Z={sim_z.round(3).tolist()};\n'
    )
    nn_colors_js = "['#f78166','#ffd93d','#7ee787','#58a6ff','#d2a8ff']"

    return f"""
<script>
(function(){{
{pts_js}
var NN_COLORS={nn_colors_js};
var K=5;

function getSimDiv(cb){{
    var divs=document.querySelectorAll('.plotly-graph-div');
    var gd=divs[divs.length-1];
    if(gd&&gd._fullLayout){{cb(gd);return;}}
    setTimeout(function(){{getSimDiv(cb);}},300);
}}

getSimDiv(function(gd){{
    var anchor=null;

    gd.on('plotly_click',function(data){{
        var pt=data.points[0];
        if(pt.curveNumber!==0) return;
        anchor={{x:pt.x, y:pt.y, z:pt.z}};

        Plotly.restyle(gd,{{
            x:[[anchor.x]], y:[[anchor.y]], z:[[anchor.z]],
            text:[['📍 ('+anchor.x.toFixed(2)+', '+anchor.y.toFixed(2)+', '+anchor.z.toFixed(2)+')']]
        }},1);

        var dists=[];
        for(var i=0;i<PTS_X.length;i++){{
            var dx=PTS_X[i]-anchor.x,
                dy=PTS_Y[i]-anchor.y,
                dz=PTS_Z[i]-anchor.z;
            dists.push({{i:i, d:Math.sqrt(dx*dx+dy*dy+dz*dz)}});
        }}
        dists.sort(function(a,b){{return a.d-b.d;}});
        var nn=dists.slice(1,K+1);

        var rows='';
        for(var k=0;k<K;k++){{
            var nb=nn[k];
            var nx=PTS_X[nb.i], ny=PTS_Y[nb.i], nz=PTS_Z[nb.i];
            var label=nb.d.toFixed(3)+' m';
            Plotly.restyle(gd,{{
                x:[[anchor.x,nx]], y:[[anchor.y,ny]], z:[[anchor.z,nz]],
                text:[['',' '+label]]
            }},k+2);
            rows+='<tr style="border-bottom:1px solid #1e2433">'+
                '<td style="padding:5px 10px;color:'+NN_COLORS[k]+';font-weight:bold">NN-'+(k+1)+'</td>'+
                '<td style="padding:5px 10px;color:#8b949e">('+nx.toFixed(2)+', '+ny.toFixed(2)+', '+nz.toFixed(2)+')</td>'+
                '<td style="padding:5px 10px;color:#ffd93d;font-weight:bold">'+label+'</td>'+
                '</tr>';
        }}
        document.getElementById('sim-tbody').innerHTML=rows;
        document.getElementById('sim-anchor').innerHTML=
            '<span style="color:{ACC3}">📍 Anchor:</span> ('+anchor.x.toFixed(3)+', '+
            anchor.y.toFixed(3)+', '+anchor.z.toFixed(3)+') m';
        document.getElementById('sim-status').textContent='✅ Anchor set — showing 5 nearest neighbours';
    }});

    window._simClear=function(){{
        anchor=null;
        Plotly.restyle(gd,{{x:[[]],y:[[]],z:[[]]}},1);
        for(var k=0;k<K;k++) Plotly.restyle(gd,{{x:[[]],y:[[]],z:[[]]}},k+2);
        document.getElementById('sim-tbody').innerHTML='';
        document.getElementById('sim-anchor').innerHTML='No anchor set';
        document.getElementById('sim-status').textContent='Click any point in the scene above to set anchor';
    }};
}});
}})();
</script>

<div style="margin:12px auto;max-width:860px;background:{BG2};
            border:1px solid {GRID};border-radius:10px;padding:16px;
            font-family:monospace;color:{TEXT};">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px">
    <span style="color:{ACC1};font-size:15px;font-weight:bold">🎯 Distance Simulator</span>
    <div style="display:flex;gap:8px">
      <span id="sim-status" style="font-size:11px;color:#8b949e;align-self:center">
        Click any point in the scene above to set anchor
      </span>
      <button onclick="_simClear()"
              style="background:{BG};color:#8b949e;border:1px solid {GRID};
                     border-radius:6px;padding:5px 14px;cursor:pointer;font-size:12px">
        ↺ Reset
      </button>
    </div>
  </div>
  <div id="sim-anchor"
       style="background:{BG};border-radius:6px;padding:9px 12px;
              font-size:12px;color:#8b949e;margin-bottom:10px;
              border-left:3px solid {ACC3}">
    No anchor set
  </div>
  <div style="max-height:190px;overflow-y:auto;border-radius:6px;border:1px solid {GRID}">
    <table style="width:100%;font-size:12px;border-collapse:collapse">
      <thead>
        <tr style="background:{BG};color:#8b949e;font-size:11px">
          <th style="padding:6px 10px;text-align:left;border-bottom:1px solid {GRID}">Rank</th>
          <th style="padding:6px 10px;text-align:left;border-bottom:1px solid {GRID}">3D coords (x, y, z) m</th>
          <th style="padding:6px 10px;text-align:left;border-bottom:1px solid {GRID}">Euclidean distance</th>
        </tr>
      </thead>
      <tbody id="sim-tbody"></tbody>
    </table>
  </div>
  <div style="margin-top:9px;font-size:10px;color:#3d4451">
    Perspective back-projected from EfficientNet-B5 + UNet++ depth | NYU Depth V2 intrinsics (fx=518.8, fy=519.0)
  </div>
</div>
""".replace('{BG}', BG).replace('{BG2}', BG2).replace('{GRID}', GRID) \
   .replace('{TEXT}', TEXT).replace('{ACC1}', ACC1) \
   .replace('{ACC3}', ACC3)


# ══════════════════════════════════════════════════════════════════════════════
# HEADER HTML  (shown once at top)
# ══════════════════════════════════════════════════════════════════════════════
def build_header_html(image_path, img_rgb, depth_m, region_stats):
    H, W = depth_m.shape
    mean_d   = float(depth_m.mean())
    med_d    = float(np.median(depth_m))
    std_d    = float(depth_m.std())
    min_d    = float(depth_m.min())
    max_d    = float(depth_m.max())
    basename = os.path.basename(image_path)

    def stat_card(label, val, unit='', color=ACC1):
        return f"""
        <div style="background:{BG2};border:1px solid {GRID};border-radius:8px;
                    padding:10px 16px;text-align:center;min-width:100px">
          <div style="font-size:10px;color:#8b949e;margin-bottom:4px">{label}</div>
          <div style="font-size:20px;font-weight:bold;color:{color}">{val}</div>
          <div style="font-size:10px;color:#8b949e">{unit}</div>
        </div>"""

    cards = (
        stat_card('Resolution', f'{W}×{H}', 'px') +
        stat_card('Mean depth', f'{mean_d:.2f}', 'm', ACC2) +
        stat_card('Median depth', f'{med_d:.2f}', 'm', ACC2) +
        stat_card('Std dev', f'{std_d:.2f}', 'm', ACC4) +
        stat_card('Min depth', f'{min_d:.2f}', 'm', '#58a6ff') +
        stat_card('Max depth', f'{max_d:.2f}', 'm', ACC3)
    )

    return f"""
<div style="background:{BG};border:1px solid {GRID};border-radius:12px;
            padding:20px 24px;margin:8px 0 16px;font-family:monospace">
  <div style="display:flex;justify-content:space-between;align-items:flex-start">
    <div>
      <div style="font-size:18px;font-weight:bold;color:{ACC1};margin-bottom:4px">
        🔭 Monocular Depth Estimation — Interactive Analysis Dashboard
      </div>
      <div style="font-size:12px;color:#8b949e">
        Model: <span style="color:{ACC2}">EfficientNet-B5 + UNet++ (SCSE attention)</span> &nbsp;|&nbsp;
        Dataset: <span style="color:{ACC2}">NYU Depth V2</span> &nbsp;|&nbsp;
        Image: <span style="color:{ACC4}">{basename}</span>
      </div>
    </div>
    <div style="font-size:11px;color:#3d4451;text-align:right">
      10 novel analysis panels<br>fully inline · no external files
    </div>
  </div>
  <div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:16px">
    {cards}
  </div>
</div>
"""


# ══════════════════════════════════════════════════════════════════════════════
# MAIN DASHBOARD  — run_dashboard()
# ══════════════════════════════════════════════════════════════════════════════
def run_dashboard(image_path: str,
                  fx: float = 518.8, fy: float = 519.0,
                  step: int = 4,
                  colormaps: tuple = ('turbo', 'jet', 'viridis'),
                  max_depth_m: float = 10.0):
    """
    Renders 6 panels directly inside the Kaggle/Jupyter notebook.

    Parameters
    ----------
    image_path  : path to a *_colors.png from NYU Depth V2 test set
    fx, fy      : camera focal lengths in pixels (NYU default)
    step        : point-cloud sub-sampling stride (lower = denser, slower)
    colormaps   : exactly 3 colormaps from CV2_CMAPS keys
    max_depth_m : depth scale factor (10 m for NYU Depth V2)
    """

    sep = '═' * 65
    print(f'\n{sep}')
    print(f'  📁  Image : {image_path}')
    print(f'{sep}')

    # ── STEP 1: inference ─────────────────────────────────────────────────
    img_rgb, depth_m, raw_norm = predict_depth(image_path, max_depth_m)
    H, W    = depth_m.shape
    d_min   = float(depth_m.min())
    d_max   = float(depth_m.max())

    # ── STEP 2: derived maps ──────────────────────────────────────────────
    print('\n  🔬  Computing edge saliency …')
    edge_sal = compute_edge_saliency(depth_m)

    print('  🔬  Estimating surface normals …')
    normals     = compute_surface_normals(depth_m, fx=fx, fy=fy)
    normals_rgb = normals_to_rgb(normals)

    print('  📊  Computing region statistics …')
    region_stats = compute_region_stats(depth_m)

    # ── STEP 3: colourmap images ──────────────────────────────────────────
    cmap_imgs = {c: depth_to_rgb(depth_m, c) for c in colormaps}

    # ── STEP 4: back-project 3-D clouds (vectorised) ─────────────────────
    clouds = {}
    print()
    for c in colormaps:
        print(f'  🌐  Back-projecting [{c}] …')
        clouds[c] = backproject(depth_m, cmap_imgs[c],
                                fx=fx, fy=fy, step=step)
        print(f'       → {len(clouds[c]["x"]):,} points')

    # ── STEP 5: build all panels ──────────────────────────────────────────
    print(f'\n{sep}')
    print('  📐  Building dashboard panels …')
    print(f'{sep}')

    fig_a = build_panel_a(img_rgb, depth_m, colormaps, d_min, d_max)
    fig_b = build_panel_b(img_rgb, depth_m, edge_sal, normals_rgb)
    fig_c = build_panel_c(img_rgb, depth_m)
    fig_d = build_panel_d(clouds, colormaps)
    fig_e = build_panel_e(depth_m, region_stats)
    fig_f, sim_x, sim_y, sim_z = build_panel_f(clouds, colormaps)
    sim_js = build_sim_js(sim_x, sim_y, sim_z)

    # ── STEP 6: render inline ─────────────────────────────────────────────
    header_html = build_header_html(image_path, img_rgb, depth_m, region_stats)

    print('\n  🖥️   Rendering header …')
    display(HTML(header_html))

    print('  🎨  Rendering Panel A: Depth maps + colormaps …')
    display(HTML(fig_a.to_html(full_html=False, include_plotlyjs='cdn')))

    print('  🔬  Rendering Panel B: Edge saliency + normals + distribution …')
    display(HTML(fig_b.to_html(full_html=False, include_plotlyjs=False)))

    print('  📐  Rendering Panel C: Cross-section profiler …')
    display(HTML(fig_c.to_html(full_html=False, include_plotlyjs=False)))

    print('  🌐  Rendering Panel D: Interactive 3-D point clouds …')
    display(HTML(fig_d.to_html(full_html=False, include_plotlyjs=False)))

    print('  📊  Rendering Panel E: Per-region statistics …')
    display(HTML(fig_e.to_html(full_html=False, include_plotlyjs=False)))

    print('  🎯  Rendering Panel F: Distance simulator …')
    sim_html = fig_f.to_html(full_html=False, include_plotlyjs=False)
    sim_html += sim_js
    display(HTML(sim_html))

    # ── STEP 7: print usage guide ─────────────────────────────────────────
    print(f'\n{"✅  All panels rendered!":^65}')
    print(f'{sep}')
    print('  HOW TO USE THIS DASHBOARD')
    print(f'  {"─"*57}')
    print('  Panel A │ Hover heatmaps for pixel+depth value')
    print('  Panel B │ Edge saliency = depth discontinuities (object edges)')
    print('         │ Surface normals = scene geometry orientation')
    print('         │ Histogram/CDF = full depth distribution analysis')
    print('  Panel C │ Horizontal/vertical depth slices through scene')
    print('  Panel D │ Drag to rotate · Scroll to zoom · Hover for coords')
    print('  Panel E │ 3×3 spatial grid mean/median/std statistics')
    print('  Panel F │ Click a 3-D point → red anchor → 5 nearest')
    print('          │ neighbours drawn with coloured lines + distances')
    print('          │ ↺ Reset clears anchor and lines')
    print(f'{sep}\n')


# ──────────────────────────────────────────────────────────────────────────────
# DISCOVER TEST IMAGES & RUN
# ──────────────────────────────────────────────────────────────────────────────
TEST_DIR    = "/kaggle/input/nyu-depth-v2/nyu_data/data/nyu2_test"
test_images = sorted(glob.glob(os.path.join(TEST_DIR, "*_colors.png")))

if not test_images:
    raise FileNotFoundError(
        f"No '*_colors.png' images found in {TEST_DIR}\n"
        "Check that the NYU Depth V2 dataset is attached to this notebook."
    )

print(f"  Found {len(test_images)} test images in {TEST_DIR}")

# ── change index (0 … len-1) to test different images ────────────────────────
IMAGE_PATH = test_images[0]

run_dashboard(
    image_path  = IMAGE_PATH,
    fx          = 518.8,
    fy          = 519.0,
    step        = 4,           # 4 = good balance; use 3 for denser clouds
    colormaps   = ('turbo', 'jet', 'viridis'),
    max_depth_m = 10.0,        # NYU Depth V2 max range
)